# LFM IBM/Graha Semantic Segmentation Inference

Run full-scene semantic-segmentation inference with a Lightning checkpoint produced by `semantic_ibm_train.ipynb`. This notebook rebuilds the same Graha model and preprocessing pipeline, loads the fine-tuned weights, predicts paired WAC/static pipeline datacubes with overlapping windows, and writes georeferenced mask and probability GeoTIFFs.

## Imports and repository setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import logging
logging.getLogger("rasterio._env").setLevel(logging.ERROR)

import sys
from pathlib import Path

import torch
from lightning.pytorch import seed_everything

In [ ]:
repo_root = Path.cwd().parent
repo_root = Path(str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup'))
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
    raise FileNotFoundError(
        "Cannot find lfm/. Run this notebook from the lfm/notebooks directory."
    )

sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.utils import create_timestamped_output_dir
from lfm.all_models.sem_seg import build_graha_notebook_configs
from lfm.full_model.sem_seg import semantic_graha_components
from lfm.full_model.sem_seg.data_cube_inference import run_datacube_inference

print("Successfully imported LFM modules")

## User configuration

Set `LIGHTNING_CHECKPOINT` to a `.ckpt` file written by `semantic_ibm_train.ipynb`, normally under `outputs/semantic_seg_finetuning/<timestamp>/checkpoints/graha_model/`. `PRETRAIN_DIR` and `DATA_DICT` must match the training run because they define the model and normalization contract. `INPUT_ROOT_DIR` is a pipeline full-scene directory containing paired WAC and Static GeoTIFFs; it does not need train/val/test folders.

In [ ]:
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "semantic_ibm_inference"
INPUT_ROOT_DIR = Path(
    "/explore/nobackup/projects/lfm/model_inputs/inference/WAC_Processed_AOI"
)
PRETRAIN_DIR = "/explore/nobackup/projects/lfm/ibm_model_pretrain_dir_v2"
LIGHTNING_CHECKPOINT = Path(
    "/path/to/semantic_seg_finetuning/checkpoints/graha_model/model-epoch-00.ckpt"
)

DATA_DICT = {
    "dataset_name": "wac_static_craters",
    "data_dir": "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/fm_all_static_all_wac_iseg_v3",
    "dataset_modality": "wac_static",
    "selected_modalities": ["vis", "uv", "static"],
    "band_filters": {
        "vis": [0, 1, 2, 3, 4],
        "uv": [0, 1],
        "static": list(range(63)),
    },
    "excluded_nodata_values": [
        -32768.0,
        -3.4028226550889045e38,
        -3.4028230607370965e38,
        -3.4028234663852886e38,
    ],
}

TILE_SIZE = 256  # Must match the spatial size used for Graha training
TILE_OVERLAP = 0.25
THRESHOLD = 0.5
INFERENCE_BATCH_SIZE = 1  # Increase if GPU memory permits
BATCH_SIZE = 8  # Retained only as part of the matching training config

# These architecture/optimizer values mirror semantic_ibm_train.ipynb.
GRAHA_BACKBONE_LR = 5.0e-5
GRAHA_HEAD_LR = 2.0e-4
GRAHA_LAYER_DECAY = 0.75
GRAHA_WEIGHT_DECAY = 0.05
GRAHA_WARMUP_STEPS = 500
GRAHA_FREEZE_BACKBONE = False
GRAHA_SHAPE_LOSS_WEIGHT = 0.05
GRAHA_SHAPE_LOSS_PAD_FRAC = 0.3

## Validate inputs and build the inference configuration

In [ ]:
if not LIGHTNING_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {LIGHTNING_CHECKPOINT}\n"
        "Set LIGHTNING_CHECKPOINT to a .ckpt produced by semantic_ibm_train.ipynb."
    )
if not INPUT_ROOT_DIR.is_dir():
    raise FileNotFoundError(f"Full-scene input directory not found: {INPUT_ROOT_DIR}")

OUTPUT_DIR = create_timestamped_output_dir(BASE_OUTPUT_DIR)
notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR,
    base_output_dir=OUTPUT_DIR,
    graha_base_output_dir=OUTPUT_DIR,
    graha_pretrain_dir=PRETRAIN_DIR,
    graha_lightning_checkpoint=LIGHTNING_CHECKPOINT,
    data_dict=DATA_DICT,
    max_epochs=1,
    graha_batch_size=BATCH_SIZE,
    max_train_samples=None,
    max_val_samples=None,
    max_test_samples=None,
    graha_backbone_lr=GRAHA_BACKBONE_LR,
    graha_head_lr=GRAHA_HEAD_LR,
    graha_layer_decay=GRAHA_LAYER_DECAY,
    graha_weight_decay=GRAHA_WEIGHT_DECAY,
    graha_warmup_steps=GRAHA_WARMUP_STEPS,
    graha_freeze_backbone=GRAHA_FREEZE_BACKBONE,
    graha_shape_loss_weight=GRAHA_SHAPE_LOSS_WEIGHT,
    graha_shape_loss_pad_frac=GRAHA_SHAPE_LOSS_PAD_FRAC,
)

config = notebook_configs.experiment_config
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies
seed_everything(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Checkpoint: {LIGHTNING_CHECKPOINT}")
print(f"Full-scene input: {INPUT_ROOT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Device: {device}")

## Rebuild the Graha task and load fine-tuned weights

In [ ]:
means, stds = semantic_graha_components.get_normalization_stats(
    graha_config, datamodule_cls=None
)
num_channels = len(config.band_filter)
sample_batch = {
    "image": torch.zeros(1, num_channels, TILE_SIZE, TILE_SIZE),
    "mask": torch.zeros(1, TILE_SIZE, TILE_SIZE, dtype=torch.long),
}

task_cls = semantic_graha_components.make_downstream_shape_segmentation_task_class(
    deps["LunarShapeSegmentationTask"]
)
graha_task = semantic_graha_components.create_task(
    graha_config, task_cls, sample_batch
).to(device)
semantic_graha_components.load_lightning_checkpoint_state(
    graha_task, LIGHTNING_CHECKPOINT, "graha"
)
graha_task.eval()
print("Graha model is ready for inference.")

## Run inference and save predictions

In [ ]:
results = run_datacube_inference(
    model=graha_task,
    device=device,
    input_dir=INPUT_ROOT_DIR,
    output_dir=OUTPUT_DIR,
    band_filter=config.band_filter,
    means=means,
    stds=stds,
    excluded_nodata_values=config.excluded_nodata_values or (),
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    threshold=THRESHOLD,
    batch_size=INFERENCE_BATCH_SIZE,
)
print(f"Completed {len(results)} full-scene tile(s).")
for result in results:
    print(result["mask"])

In [ ]:
results

## Cleanup

In [ ]:
del graha_task
if torch.cuda.is_available():
    torch.cuda.empty_cache()